In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# ==========================================
# 1. LOAD DATASET & PELATIHAN MODEL
# ==========================================
train_df = pd.read_csv('Train_Data1_20pct.csv', sep=';')
test_df = pd.read_csv('Test_Data1_20pct.csv', sep=';')

fitur_sensor = [
    'pm25_concentration_ugm3',
    'pm10_concentration_ugm3',
    'carbon_monoxide_co_ugm3',
    'sulfur_dioxide_so2_ugm3',
    'nitrogen_dioxide_no2_ugm3',
    'ozone_o3_ugm3',
    'ambient_temperature_celsius',
    'relative_humidity_pct',
    'wind_speed_kmh',
    'rainfall_intensity_mm'
]

train_clean = train_df[fitur_sensor + ['Target_Encoded']].dropna()
test_clean = test_df[fitur_sensor + ['Target_Encoded']].dropna()

X_train = train_clean[fitur_sensor]
y_train = train_clean['Target_Encoded'].astype(int)
X_test = test_clean[fitur_sensor]
y_test = test_clean['Target_Encoded'].astype(int)

label_kategori = {
    0: 'Baik',
    1: 'Berbahaya',
    2: 'Sangat Tidak Sehat',
    3: 'Sedang',
    4: 'Tidak Sehat'
}

rf_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
svm_model = SVC(kernel='rbf', C=1.0, random_state=42).fit(X_train, y_train)

acc_rf = accuracy_score(y_test, rf_model.predict(X_test)) * 100
acc_svm = accuracy_score(y_test, svm_model.predict(X_test)) * 100

# ==========================================
# 2. DASHBOARD INTERAKTIF GOOGLE COLAB
# ==========================================
custom_css = widgets.HTML("""
<style>
    .ind-header-box {
        background: linear-gradient(145deg, #0b1120 0%, #111827 100%);
        border: 1px solid #1e293b;
        border-radius: 10px;
        padding: 16px 22px;
        display: flex;
        justify-content: space-between;
        align-items: center;
        font-family: 'Segoe UI', sans-serif;
        margin-bottom: 10px;
    }
    .ind-section-title {
        font-size: 12px;
        font-weight: 700;
        letter-spacing: 1px;
        color: #38bdf8;
        border-left: 3px solid #0284c7;
        padding-left: 8px;
        margin-bottom: 8px;
    }
    .widget-label { color: #e2e8f0 !important; font-size: 12.5px !important; }
    .widget-readout { color: #38bdf8 !important; font-weight: bold !important; font-family: monospace !important; }
</style>
""")

header_ui = widgets.HTML(f"""
<div class="ind-header-box">
    <div>
        <div style="font-size: 19px; font-weight: 700; color: #f8fafc; letter-spacing: 0.8px;">
            AIR QUALITY TELEMETRY & AI DIAGNOSTICS
        </div>
        <div style="font-size: 12px; color: #94a3b8; margin-top: 3px;">
            INDUSTRIAL MONITORING SYSTEM (Z-SCORE TELEMETRY) &nbsp;|&nbsp; RF Accuracy: <b>{acc_rf:.2f}%</b> &nbsp;•&nbsp; SVM Accuracy: <b>{acc_svm:.2f}%</b>
        </div>
    </div>
    <div style="background: rgba(16, 185, 129, 0.15); color: #34d399; border: 1px solid #059669; padding: 5px 12px; border-radius: 20px; font-size: 11px; font-weight: 700;">
        ● SYSTEM ONLINE
    </div>
</div>
""")

label_map = {
    'pm25_concentration_ugm3': 'PM2.5 (Z-score)',
    'pm10_concentration_ugm3': 'PM10 (Z-score)',
    'carbon_monoxide_co_ugm3': 'Gas CO (Z-score)',
    'sulfur_dioxide_so2_ugm3': 'Gas SO₂ (Z-score)',
    'nitrogen_dioxide_no2_ugm3': 'Gas NO₂ (Z-score)',
    'ozone_o3_ugm3': 'Gas O₃ (Z-score)',
    'ambient_temperature_celsius': 'Suhu (Z-score)',
    'relative_humidity_pct': 'Kelembapan (Z)',
    'wind_speed_kmh': 'Angin (Z-score)',
    'rainfall_intensity_mm': 'Hujan (Z-score)'
}

sliders = {}
for col in fitur_sensor:
    sliders[col] = widgets.FloatSlider(
        value=float(X_train[col].median()),
        min=float(X_train[col].min()),
        max=float(X_train[col].max()),
        step=0.05,
        description=label_map[col],
        readout_format='.2f',
        continuous_update=False,
        style={'description_width': '135px'},
        layout=widgets.Layout(width='98%', margin='3px 0px')
    )

panel_kiri = widgets.VBox(
    [widgets.HTML('<div class="ind-section-title">PARAMETER GAS & PARTIKULAT</div>')] +
    [sliders[c] for c in fitur_sensor[:6]],
    layout=widgets.Layout(width='50%', padding='12px', border='1px solid #1e293b', margin='0 6px 0 0')
)

panel_kanan = widgets.VBox(
    [widgets.HTML('<div class="ind-section-title">PARAMETER METEOROLOGI</div>')] +
    [sliders[c] for c in fitur_sensor[6:]],
    layout=widgets.Layout(width='50%', padding='12px', border='1px solid #1e293b', margin='0 0 0 6px')
)

# Tombol Skenario & Simpan PNG
btn_baik = widgets.Button(description='Udara Bersih', button_style='info', layout=widgets.Layout(width='130px', height='34px'))
btn_buruk = widgets.Button(description='Polusi Tinggi', button_style='warning', layout=widgets.Layout(width='130px', height='34px'))
btn_bahaya = widgets.Button(description='Darurat Asap', button_style='danger', layout=widgets.Layout(width='130px', height='34px'))
btn_save = widgets.Button(description='Simpan PNG', button_style='primary', icon='download', layout=widgets.Layout(width='145px', height='34px'))

status_simpan = widgets.HTML(value="")
output_panel = widgets.Output()

status_colors = {
    'Baik': ('#10b981', 'rgba(16, 185, 129, 0.15)', 'AMAN — Kualitas udara sangat baik, tidak menimbulkan risiko kesehatan.'),
    'Sedang': ('#3b82f6', 'rgba(59, 130, 246, 0.15)', 'MODERAT — Masih dapat diterima, waspada bagi kelompok sensitif.'),
    'Tidak Sehat': ('#f59e0b', 'rgba(245, 158, 11, 0.15)', 'PERINGATAN — Mulai berdampak pada kesehatan pernapasan masyarakat.'),
    'Sangat Tidak Sehat': ('#ef4444', 'rgba(239, 68, 68, 0.15)', 'BAHAYA TINGGI — Segera aktifkan sistem filtrasi udara & batasi aktivitas luar.'),
    'Berbahaya': ('#c084fc', 'rgba(168, 85, 247, 0.20)', 'DARURAT LINGKUNGAN — Kondisi kritis, evakuasi / proteksi pernapasan wajib.')
}

def evaluasi_realtime(change=None):
    with output_panel:
        clear_output(wait=True)
        vals = [[sliders[col].value for col in fitur_sensor]]
        df_in = pd.DataFrame(vals, columns=fitur_sensor)

        pred_rf = label_kategori[int(rf_model.predict(df_in)[0])]
        pred_svm = label_kategori[int(svm_model.predict(df_in)[0])]
        prob_rf = rf_model.predict_proba(df_in).max() * 100

        col_rf, bg_rf, desc_rf = status_colors[pred_rf]
        col_svm, bg_svm, _ = status_colors[pred_svm]

        display(HTML(f"""
        <div style="background: #0f172a; border: 1px solid #334155; border-radius: 10px; padding: 16px 20px; margin-top: 8px; font-family: 'Segoe UI', sans-serif;">
            <div style="font-size: 11px; color: #94a3b8; letter-spacing: 1px; font-weight: 700; margin-bottom: 10px;">
                REAL-TIME AI DIAGNOSTIC OUTPUT
            </div>
            <div style="display: flex; gap: 16px; flex-wrap: wrap;">
                <div style="flex: 1; min-width: 240px; background: {bg_rf}; border-left: 5px solid {col_rf}; padding: 12px 16px; border-radius: 6px;">
                    <div style="font-size: 11px; color: #cbd5e1; font-weight: 600;">PRIMARY ENGINE: RANDOM FOREST</div>
                    <div style="font-size: 22px; font-weight: 800; color: {col_rf}; margin: 4px 0;">{pred_rf.upper()}</div>
                    <div style="font-size: 12px; color: #e2e8f0;">Confidence Score: <b>{prob_rf:.1f}%</b></div>
                </div>
                <div style="flex: 1; min-width: 240px; background: {bg_svm}; border-left: 5px solid {col_svm}; padding: 12px 16px; border-radius: 6px;">
                    <div style="font-size: 11px; color: #cbd5e1; font-weight: 600;">SECONDARY ENGINE: SVM (RBF KERNEL)</div>
                    <div style="font-size: 22px; font-weight: 800; color: {col_svm}; margin: 4px 0;">{pred_svm.upper()}</div>
                    <div style="font-size: 12px; color: #e2e8f0;">Margin Classifier Status: <b>Active</b></div>
                </div>
            </div>
            <div style="margin-top: 12px; padding-top: 10px; border-top: 1px solid #1e293b; font-size: 12.5px; color: #f8fafc;">
                <b>Rekomendasi Sistem:</b> {desc_rf}
            </div>
        </div>
        """))

# Hubungkan semua slider agar otomatis memprediksi saat digeser di Colab
for sl in sliders.values():
    sl.observe(evaluasi_realtime, names='value')

def apply_preset(vals):
    status_simpan.value = ""
    for col, v in zip(fitur_sensor, vals):
        sliders[col].value = v

# Fungsi ekspor snapshot dashboard ke file PNG di Colab
def simpan_gambar_colab(b):
    vals = [sliders[col].value for col in fitur_sensor]
    df_in = pd.DataFrame([vals], columns=fitur_sensor)
    pred_rf = label_kategori[int(rf_model.predict(df_in)[0])]
    pred_svm = label_kategori[int(svm_model.predict(df_in)[0])]
    prob_rf = rf_model.predict_proba(df_in).max() * 100
    col_rf, desc_rf = status_colors[pred_rf][0], status_colors[pred_rf][2]
    col_svm = status_colors[pred_svm][0]

    fig, ax = plt.subplots(figsize=(11, 5.5), facecolor='#0b1120')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    ax.text(0.03, 0.91, 'AIR QUALITY TELEMETRY & AI DIAGNOSTICS', fontsize=15, fontweight='bold', color='#f8fafc')
    ax.text(0.03, 0.85, f'RF Accuracy: {acc_rf:.2f}%  |  SVM Accuracy: {acc_svm:.2f}%', fontsize=9.5, color='#94a3b8')

    # Tampilkan nilai 10 parameter sensor saat ini
    for idx, (col, val) in enumerate(zip(fitur_sensor, vals)):
        x_pos = 0.03 if idx < 5 else 0.52
        y_pos = 0.74 - ((idx % 5) * 0.065)
        ax.text(x_pos, y_pos, f"• {label_map[col]:<18}: {val:.2f}", fontsize=10, color='#e2e8f0', family='monospace')

    # Kotak Hasil Prediksi RF & SVM
    ax.add_patch(FancyBboxPatch((0.03, 0.14), 0.44, 0.22, boxstyle="round,pad=0.02", facecolor='#1e293b', edgecolor=col_rf, linewidth=2))
    ax.text(0.05, 0.30, 'RANDOM FOREST PREDICTION', fontsize=8.5, fontweight='bold', color='#cbd5e1')
    ax.text(0.05, 0.21, pred_rf.upper(), fontsize=16, fontweight='bold', color=col_rf)
    ax.text(0.05, 0.155, f'Confidence: {prob_rf:.1f}%', fontsize=9, color='#e2e8f0')

    ax.add_patch(FancyBboxPatch((0.52, 0.14), 0.45, 0.22, boxstyle="round,pad=0.02", facecolor='#1e293b', edgecolor=col_svm, linewidth=2))
    ax.text(0.54, 0.30, 'SVM (RBF) PREDICTION', fontsize=8.5, fontweight='bold', color='#cbd5e1')
    ax.text(0.54, 0.21, pred_svm.upper(), fontsize=16, fontweight='bold', color=col_svm)
    ax.text(0.54, 0.155, 'Status: Active', fontsize=9, color='#e2e8f0')

    ax.text(0.03, 0.05, f'Rekomendasi: {desc_rf}', fontsize=9.5, fontweight='bold', color='#f8fafc')

    nama_file = f"Dashboard_Colab_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
    fig.savefig(nama_file, dpi=300, facecolor='#0b1120', bbox_inches='tight')
    plt.close(fig)
    status_simpan.value = f"<span style='color:#34d399; font-weight:bold; margin-left:10px;'>✓ Tersimpan di panel Files Colab: {nama_file}</span>"

btn_baik.on_click(lambda b: apply_preset([-1.15, -1.22, -1.10, -0.90, -1.32, -0.44, 0.0, 0.0, 0.5, 0.5]))
btn_buruk.on_click(lambda b: apply_preset([0.80, 0.85, 0.50, 0.20, 0.60, 0.10, 0.5, -0.3, -0.5, -0.37]))
btn_bahaya.on_click(lambda b: apply_preset([5.50, 5.30, 3.20, 0.30, -0.50, 0.05, 1.0, -0.5, -1.0, -0.37]))
btn_save.on_click(simpan_gambar_colab)

# Render seluruh antarmuka di Google Colab
toolbar = widgets.HBox([btn_baik, btn_buruk, btn_bahaya, btn_save, status_simpan], layout=widgets.Layout(margin='0 0 10px 0', align_items='center'))
grid = widgets.HBox([panel_kiri, panel_kanan], layout=widgets.Layout(width='100%'))

display(custom_css)
display(header_ui)
display(toolbar)
display(grid)
display(output_panel)

evaluasi_realtime()

HTML(value="\n<style>\n    .ind-header-box {\n        background: linear-gradient(145deg, #0b1120 0%, #111827 …

HTML(value='\n<div class="ind-header-box">\n    <div>\n        <div style="font-size: 19px; font-weight: 700; …

Output()